# Example 7.9.1 / 2: Ask the Solver for a Feasibility Relaxation

Companion notebook for **7.9.1 Feasibility Relaxation** in the AMPL lab Documentation, section "Ask the Solver for a Feasibility Relaxation". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Ask the Solver for a Feasibility Relaxation

Some MP drivers build the relaxation themselves with the option `alg:feasrelax`: `1` minimizes the
weighted sum of violations, `2` the sum of squares, `3` the weighted count of violated rows, and
`4` to `6` the same followed by the original objective. Weights come from the suffixes `.rhspen`
on constraints and `.lbpen`, `.ubpen` on variable bounds, or default to 1. On the original
infeasible model, with AMPL's presolve turned off so that it does not stop the run before the
solver sees the model:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                         # products
param rate {PROD} > 0;            # tons produced per hour
param profit {PROD};              # profit per ton
param market {PROD} >= 0;         # most that can be sold, in tons
param commit {PROD} >= 0;         # tons already ordered
param avail >= 0;                 # hours available

var Make {p in PROD} >= 0, <= market[p];  # tons produced
maximize TotalProfit: sum {p in PROD} profit[p] * Make[p];
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;  # hours available
subject to Orders {p in PROD}: Make[p] >= commit[p];                # firm orders

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["commit"] = {"bands": 4000, "coils": 2000, "plate": 1500}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.option["presolve"] = 0
ampl.solve(solver="gurobi", gurobi_options="alg:feasrelax=1")
print("hours used:", ampl.con["Time"].body(), "| Make:", ampl.var["Make"].to_dict())

Gurobi 13.0.0:   alg:feasrelax = 1
Gurobi 13.0.0: optimal solution; feasrelax objective 3.660714286
1 simplex iteration
 
------------ WARNINGS ------------
  Type                         MaxAbs [Name]   MaxRel [Name]
  objective(s)                 2E+05           1E+00         
* algebraic con(s)             4E+00           9E-02         
*: Using the solver's aux variable values.
Documentation: mp.ampl.com/modeling-tools.html#automatic-solution-check.
hours used: 43.660714285714285 | Make: {'bands': 4000, 'coils': 2000, 'plate': 1500}
